## Limpieza de datos

In [ ]:
# ==========================================
# NOTEBOOK: 00_limpieza_datos.ipynb
# OBJETIVO:
# - Cargar los 3 datasets desde data/raw
# - Limpiarlos
# - Guardarlos en data/processed
# ==========================================

# ==========
# 1. LIBRERÍAS
# ==========
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path

# ==========
# 2. MOSTRAR DESDE DÓNDE SE EJECUTA EL NOTEBOOK
# Esto ayuda a saber si Jupyter está trabajando desde la raíz del proyecto
# o desde la carpeta notebooks
# ==========
print("Carpeta actual de ejecución:", os.getcwd())

# ==========
# 3. DETECTAR AUTOMÁTICAMENTE LA RUTA BASE DEL PROYECTO
# Si existe data/raw en la carpeta actual, usamos esa.
# Si no, probamos con ../data/raw
# ==========
if Path("data/raw").exists():
    BASE_PATH = Path(".")
elif Path("../data/raw").exists():
    BASE_PATH = Path("..")
else:
    raise FileNotFoundError("No se encontró la carpeta data/raw")

# ==========
# 4. DEFINIR RUTAS DE ENTRADA Y SALIDA
# ==========
RAW_PATH = BASE_PATH / "data" / "raw"
PROCESSED_PATH = BASE_PATH / "data" / "processed"

# Crear la carpeta processed si no existe
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# ==========
# 5. DEFINIR LOS NOMBRES REALES DE TUS ARCHIVOS
# OJO: en tu proyecto actual el archivo de attrition se llama
# women_tech_attrition_clean.csv y está dentro de raw
# ==========
attrition_file = RAW_PATH / "women_tech_attrition_clean.csv"
salary_file = RAW_PATH / "salarios_stem.csv"
jobs_file = RAW_PATH / "job_offers_bias.csv"

# ==========
# 6. COMPROBAR SI LOS ARCHIVOS EXISTEN
# ==========
print("\n--- Comprobación de archivos ---")
print("Ruta RAW:", RAW_PATH.resolve())
print("Attrition:", attrition_file.exists(), "-", attrition_file.name)
print("Salary:", salary_file.exists(), "-", salary_file.name)
print("Jobs:", jobs_file.exists(), "-", jobs_file.name)

# ==========
# 7. CARGAR LOS DATASETS
# Para job_offers_bias.csv usamos engine='python' y on_bad_lines='skip'
# porque ese tipo de CSV puede dar error si alguna fila tiene comas o comillas mal cerradas.
# pandas permite estas opciones para manejar líneas problemáticas. [web:72][web:65]
# ==========
df_attrition = pd.read_csv(attrition_file)
df_salary = pd.read_csv(salary_file)
df_jobs = pd.read_csv(jobs_file, engine="python", on_bad_lines="skip")

# ==========
# 8. MOSTRAR TAMAÑO INICIAL
# ==========
print("\n--- Tamaño inicial de los datasets ---")
print("Attrition:", df_attrition.shape)
print("Salary:", df_salary.shape)
print("Jobs:", df_jobs.shape)

# ==========
# 9. FUNCIONES AUXILIARES
# ==========

# Función para limpiar nombres de columnas:
# - quita espacios
# - pasa a minúsculas
# - cambia espacios por guiones bajos
def clean_column_names(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

# Función básica para limpiar texto:
# - convierte a string
# - quita espacios al inicio y final
# - pasa a minúsculas
# - reduce espacios dobles
def clean_text_basic(text):
    if pd.isna(text):
        return text
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

# Función para limpiar descripciones de ofertas
def clean_job_description(text):
    if pd.isna(text):
        return text
    text = str(text).lower().strip()
    text = re.sub(r"[\"']", "", text)      # quita comillas simples y dobles
    text = re.sub(r"\s+", " ", text)       # quita espacios dobles
    return text

# ==========================================
# 10. LIMPIEZA DEL DATASET 1: ATTRITION
# ==========================================

# Limpiar nombres de columnas
df_attrition = clean_column_names(df_attrition)

# Ver columnas
print("\nColumnas attrition:", df_attrition.columns.tolist())

# Columnas de texto que vamos a normalizar
text_cols_attrition = [
    "tamano_empresa",
    "teletrabajo",
    "politicas_conciliacion",
    "satisfaccion_laboral",
    "brecha_salarial_percibida",
    "abandona"
]

for col in text_cols_attrition:
    if col in df_attrition.columns:
        df_attrition[col] = df_attrition[col].apply(clean_text_basic)

# Columnas numéricas
num_cols_attrition = [
    "id",
    "edad",
    "anios_experiencia",
    "salario_anual",
    "anios_en_puesto"
]

for col in num_cols_attrition:
    if col in df_attrition.columns:
        df_attrition[col] = pd.to_numeric(df_attrition[col], errors="coerce")

# Eliminar duplicados
df_attrition = df_attrition.drop_duplicates()

# Eliminar nulos importantes
required_attrition = ["edad", "anios_experiencia", "salario_anual", "abandona"]
required_attrition = [col for col in required_attrition if col in df_attrition.columns]
df_attrition = df_attrition.dropna(subset=required_attrition)

# Reiniciar índice
df_attrition = df_attrition.reset_index(drop=True)

# Mostrar resumen
print("\n--- Attrition limpio ---")
print(df_attrition.shape)
print(df_attrition.isnull().sum())

# ==========================================
# 11. LIMPIEZA DEL DATASET 2: SALARIOS
# ==========================================

# Limpiar nombres de columnas
df_salary = clean_column_names(df_salary)

# Ver columnas
print("\nColumnas salary:", df_salary.columns.tolist())

# Limpiar texto
text_cols_salary = [
    "moneda",
    "genero",
    "cargo",
    "sector",
    "pais",
    "ciudad",
    "tipo_jornada"
]

for col in text_cols_salary:
    if col in df_salary.columns:
        df_salary[col] = df_salary[col].apply(clean_text_basic)

# Convertir columnas numéricas
num_cols_salary = [
    "id",
    "salario_anual",
    "anios_experiencia"
]

for col in num_cols_salary:
    if col in df_salary.columns:
        df_salary[col] = pd.to_numeric(df_salary[col], errors="coerce")

# Eliminar duplicados
df_salary = df_salary.drop_duplicates()

# Eliminar nulos importantes
required_salary = ["salario_anual", "genero", "anios_experiencia", "cargo", "sector"]
required_salary = [col for col in required_salary if col in df_salary.columns]
df_salary = df_salary.dropna(subset=required_salary)

# Reiniciar índice
df_salary = df_salary.reset_index(drop=True)

# Mostrar resumen
print("\n--- Salary limpio ---")
print(df_salary.shape)
print(df_salary.isnull().sum())

# ==========================================
# 12. LIMPIEZA DEL DATASET 3: JOB OFFERS / NLP
# ==========================================

# Limpiar nombres de columnas
df_jobs = clean_column_names(df_jobs)

# Ver columnas
print("\nColumnas jobs:", df_jobs.columns.tolist())

# Limpiar texto general
text_cols_jobs = [
    "puesto",
    "sector",
    "descripcion",
    "nivel_sesgo"
]

for col in text_cols_jobs:
    if col in df_jobs.columns:
        df_jobs[col] = df_jobs[col].apply(clean_text_basic)

# Limpieza especial para la descripción
if "descripcion" in df_jobs.columns:
    df_jobs["descripcion"] = df_jobs["descripcion"].apply(clean_job_description)

# Convertir id a numérico
if "id" in df_jobs.columns:
    df_jobs["id"] = pd.to_numeric(df_jobs["id"], errors="coerce")

# Eliminar duplicados
df_jobs = df_jobs.drop_duplicates()

# Eliminar filas con datos nulos en columnas importantes
required_jobs = ["descripcion", "nivel_sesgo"]
required_jobs = [col for col in required_jobs if col in df_jobs.columns]
df_jobs = df_jobs.dropna(subset=required_jobs)

# Reiniciar índice
df_jobs = df_jobs.reset_index(drop=True)

# Mostrar resumen
print("\n--- Jobs limpio ---")
print(df_jobs.shape)
print(df_jobs.isnull().sum())

# Si existe la etiqueta, mostrar cuántos casos hay por clase
if "nivel_sesgo" in df_jobs.columns:
    print("\nDistribución de nivel_sesgo:")
    print(df_jobs["nivel_sesgo"].value_counts())

# ==========================================
# 13. GUARDAR LOS ARCHIVOS LIMPIOS EN data/processed
# ==========================================
df_attrition.to_csv(PROCESSED_PATH / "women_tech_attrition_clean.csv", index=False)
df_salary.to_csv(PROCESSED_PATH / "salarios_stem_clean.csv", index=False)
df_jobs.to_csv(PROCESSED_PATH / "job_offers_bias_clean.csv", index=False)

# ==========================================
# 14. MOSTRAR RESULTADO FINAL
# ==========================================
print("\n--- Archivos guardados en data/processed ---")
for f in PROCESSED_PATH.glob("*"):
    print("-", f.name)

print("\n--- Vista previa: women_tech_attrition_clean ---")
display(df_attrition.head())

print("\n--- Vista previa: salarios_stem_clean ---")
display(df_salary.head())

print("\n--- Vista previa: job_offers_bias_clean ---")
display(df_jobs.head())


Carpeta actual de ejecución: c:\Users\raula\OneDrive\Documenten\voyage\Data Scientist\SomosF5\Proyecto Final\notebooks

--- Comprobación de archivos ---
Ruta RAW: C:\Users\raula\OneDrive\Documenten\voyage\Data Scientist\SomosF5\Proyecto Final\data\raw
Attrition: True - women_tech_attrition_clean.csv
Salary: True - salarios_stem.csv
Jobs: True - job_offers_bias.csv

--- Tamaño inicial de los datasets ---
Attrition: (100, 11)
Salary: (100, 10)
Jobs: (90, 5)

Columnas attrition: ['id', 'edad', 'anios_experiencia', 'salario_anual', 'tamano_empresa', 'teletrabajo', 'politicas_conciliacion', 'satisfaccion_laboral', 'brecha_salarial_percibida', 'anios_en_puesto', 'abandona']

--- Attrition limpio ---
(100, 11)
id                           0
edad                         0
anios_experiencia            0
salario_anual                0
tamano_empresa               0
teletrabajo                  0
politicas_conciliacion       0
satisfaccion_laboral         0
brecha_salarial_percibida    0
anios_en

,id,edad,anios_experiencia,salario_anual,tamano_empresa,teletrabajo,politicas_conciliacion,satisfaccion_laboral,brecha_salarial_percibida,anios_en_puesto,abandona
0,1,24,1,28000,pequena,si,no,baja,si,1,si
1,2,25,2,30000,pequena,si,no,media,si,1,si
2,3,26,3,32000,mediana,si,si,alta,no,2,no
3,4,27,4,34000,mediana,no,no,baja,si,2,si
4,5,28,5,36000,grande,si,si,alta,no,3,no



--- Vista previa: salarios_stem_clean ---


,id,salario_anual,moneda,genero,anios_experiencia,cargo,sector,pais,ciudad,tipo_jornada
0,1,24000,eur,mujer,0,junior developer,software,españa,madrid,tiempo completo
1,2,27000,eur,hombre,0,junior developer,software,españa,madrid,tiempo completo
2,3,26000,eur,mujer,1,junior developer,fintech,españa,barcelona,tiempo completo
3,4,29000,eur,hombre,1,junior developer,fintech,españa,barcelona,tiempo completo
4,5,25000,eur,no binario,1,data analyst,consultoría,españa,madrid,tiempo completo



--- Vista previa: job_offers_bias_clean ---


,id,puesto,sector,descripcion,nivel_sesgo
0,1,junior developer,software,"buscamos una persona con ganas de aprender, co...",bajo
1,2,backend developer,fintech,"se valora capacidad técnica, comunicación efec...",bajo
2,3,data analyst,consultoría,"la persona candidata analizará datos, preparar...",bajo
3,4,qa engineer,software,oferta orientada a una persona detallista que ...,bajo
4,5,frontend developer,ecommerce,"buscamos una persona creativa, organizada y co...",bajo
